In [1]:
import json, re, unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoModelForCTC, Trainer, TrainingArguments,
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
)

ROOT = Path("..")  # project root (this notebook lives in notebooks/)
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "runs" / "w2v2-geo"
MODEL_NAME = "facebook/wav2vec2-base" # "facebook/wav2vec2-xls-r-300m"
SAMPLING_RATE = 16_000
QUICK_TEST = True

/Users/nkhieu02/Desktop/ASR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1111 17:23:01.373000 3110 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
def load_split(name):
    df = pd.read_csv(DATA_DIR / "geo" / f"{name}.csv", keep_default_na=False)
    df["path"] = df["file"].map(lambda f: str(DATA_DIR / f))
    exists = df["path"].map(lambda p: Path(p).exists())
    print(f"{name}: {exists.sum()} / {len(df)} clips available")
    return df, df[exists].reset_index(drop=True)

train_all, train_df = load_split("train")
dev_all, dev_df = load_split("dev")
test_all, test_df = load_split("test")

if QUICK_TEST:
    train_df, dev_df, test_df = train_df.head(64), dev_df.head(16), test_df.head(8)
train_df.head()

train: 6000 / 6000 clips available
dev: 1000 / 1000 clips available
test: 1000 / 1000 clips available


,file,transcript,path
0,geo/clips/train_0.wav,gleizt eztez varemta la zanan zbertom,data/geo/clips/train_0.wav
1,geo/clips/train_1.wav,baldaŭe li hafez buztimum em nedicena imztetut...,data/geo/clips/train_1.wav
2,geo/clips/train_2.wav,pozt egzbludo de la bulapolŝefega milito patal...,data/geo/clips/train_2.wav
3,geo/clips/train_3.wav,zid omidiri la prublenu zaĝe eztez zolfeta,data/geo/clips/train_3.wav
4,geo/clips/train_4.wav,mi folas domi niajn zemcirajm damgujm al ili,data/geo/clips/train_4.wav


In [3]:
def normalize(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s']", "", text)
    return re.sub(r"\s+", " ", text).strip()

for df in (train_all, dev_all, train_df, dev_df):
    df["transcript"] = df["transcript"].map(normalize)

chars = sorted(set("".join(train_all["transcript"]) + "".join(dev_all["transcript"])))
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"] = vocab.pop(" ")
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)
print(len(vocab), vocab)

OUT_DIR.mkdir(exist_ok=True)
with open(OUT_DIR / "vocab.json", "w") as f:
    json.dump(vocab, f, ensure_ascii=False)

31 {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'z': 22, 'ĉ': 23, 'ĝ': 24, 'ĥ': 25, 'ĵ': 26, 'ŝ': 27, 'ŭ': 28, '|': 0, '[UNK]': 29, '[PAD]': 30}


In [4]:
tokenizer = Wav2Vec2CTCTokenizer(
    str(OUT_DIR / "vocab.json"), unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SAMPLING_RATE, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(OUT_DIR)

['w2v2-geo/processor_config.json']

In [5]:
def prepare(batch):
    audio, sr = sf.read(batch["path"], dtype="float32")
    assert sr == SAMPLING_RATE, sr
    batch["input_values"] = processor(audio, sampling_rate=sr).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor(text=batch["transcript"]).input_ids
    return batch

cols = ["path", "transcript"]
train_ds = Dataset.from_pandas(train_df[cols]).map(prepare, remove_columns=cols)
dev_ds = Dataset.from_pandas(dev_df[cols]).map(prepare, remove_columns=cols)
print(train_ds, dev_ds)

Map: 100%|██████████| 16/16 [00:00<00:00, 641.10 examples/s]

Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 64
}) Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 16
})


In [6]:
MAX_SECONDS = 15
train_ds = train_ds.filter(lambda n: n < MAX_SECONDS * SAMPLING_RATE, input_columns="input_length")

Filter: 100%|██████████| 64/64 [00:00<00:00, 18191.61 examples/s]


In [7]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: str = "longest"

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [8]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

In [9]:
model = AutoModelForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 40090.52it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=5 if QUICK_TEST else 500,
    max_steps=20 if QUICK_TEST else 4000,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    train_sampling_strategy="group_by_length",
    length_column_name="input_length",
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    eval_steps=10 if QUICK_TEST else 500,
    save_steps=10 if QUICK_TEST else 500,
    logging_steps=5 if QUICK_TEST else 25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 32, 'bos_token_id': 31}.
/Users/nkhieu02/Desktop/ASR/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Wer,Cer
10,13.541396,3.835830,1.000000,1.000000
20,6.883231,3.399436,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.86it/s]
/Users/nkhieu02/Desktop/ASR/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.26it/s]


TrainOutput(global_step=20, training_loss=16.003839683532714, metrics={'train_runtime': 112.2628, 'train_samples_per_second': 2.85, 'train_steps_per_second': 0.178, 'total_flos': 2.1465481181054976e+16, 'train_loss': 16.003839683532714, 'epoch': 5.0})

In [11]:
trainer.save_model(OUT_DIR / "best")
processor.save_pretrained(OUT_DIR / "best")
trainer.evaluate()

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.92it/s]
/Users/nkhieu02/Desktop/ASR/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Wer,Cer
6.883231,3.835830,20,1.000000,1.000000


{'eval_loss': 3.8358302116394043, 'eval_wer': 1.0, 'eval_cer': 1.0}

In [14]:
model.eval()
device = model.device

def transcribe(path):
    audio, sr = sf.read(path, dtype="float32")
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return processor.batch_decode(logits.argmax(dim=-1))[0]

for _, row in dev_df.head(5).iterrows():
    print("REF:", row["transcript"])
    print("PRED:", transcribe(row["path"]), "\n")

REF: ili gfeetekiz la turnemtetam gurun de la zimtina gmapemu
PRED:  

REF: bumaj imvamuj kebatrujm veliĉekaz nalbunaj ilin emtonpekaz
PRED:  

REF: mi vareĝiz espiramtestoj
PRED:  

REF: vernando zem eztez bervidimta
PRED:  

REF: bre la luĝamtoj de la murdzimtemela insolu omi zceaz nalnoltom
PRED:  



In [16]:
test_df["transcript"] = [transcribe(p) for p in test_df["path"]]
test_df[["file", "transcript"]].to_csv(OUT_DIR / "test_predictions.csv", index=False)
test_df[["file", "transcript"]].head()

,file,transcript
0,geo/clips/test_0.wav,
1,geo/clips/test_1.wav,
2,geo/clips/test_2.wav,
3,geo/clips/test_3.wav,
4,geo/clips/test_4.wav,
